<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_01_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 1 · versión IA · El encargo, no el código

Este es el mismo laboratorio 1, con el mismo negocio y las mismas conclusiones, resuelto de otra
manera: **tú no escribes el código, lo encargas.** Cada paso trae un encargo listo para copiar y
pegar en tu asistente, una celda vacía donde pegas lo que te devuelva, y una celda de comprobación
que te dice si el número que te salió es el correcto.

Lo que se practica hoy no es pandas. Es la habilidad que el curso califica de verdad: **pedir bien y
verificar en serio.**

> **Hoy haces** · Recorres un análisis completo de Comercial Andina (90 min) sin escribir código a
> mano: cargas dos tablas, describes lo que pasó, produces un gráfico y escribes una conclusión de
> negocio, encargándole cada pieza al asistente con el marco CLARO. Después reproduces un informe real
> con datos correctos y conclusión inválida, y lo corriges.
>
> **Entrega** · Este cuaderno ejecutado de arriba a abajo, con los seis encargos pegados y corriendo,
> los tres ejercicios resueltos y al menos tres filas propias en la bitácora de prompts. Nombre de
> archivo: `lab_01_ia_apellido.ipynb`.
>
> ⚠️ **Antes de empezar** · Ejecuta la celda de abajo. Es la única del cuaderno que ya viene escrita
> y no se encarga: prepara el entorno y descarga los datos.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
BASE_URL = ("https://raw.githubusercontent.com/mayait/"
            "CursoAnalisisDatos_IA_2026/main/sitio/datos")
ARCHIVOS = ["clientes.csv", "productos.csv", "sucursales.csv", "ventas.csv",
            "ventas_limpias.csv", "marketing_mensual.csv",
            "experimento_reactivacion.csv"]
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if (p / "ventas.csv").exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se descargan los siete archivos una vez.
    from urllib.request import urlretrieve
    DATOS = Path("datos")
    DATOS.mkdir(exist_ok=True)
    for archivo in ARCHIVOS:
        if not (DATOS / archivo).exists():
            urlretrieve(f"{BASE_URL}/{archivo}", DATOS / archivo)

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. El encargo se califica; el código no

Un asistente de IA hace exactamente lo que le pides. Ese es el problema: si el encargo es vago, la
respuesta es plausible y equivocada, y como viene bien escrita cuesta el doble darse cuenta.

En este curso los encargos se escriben con cinco partes. El marco se llama **CLARO** y las cinco
letras son las cinco preguntas que un asistente no puede adivinar por ti:

| Letra | Parte | Qué responde | Qué pasa si la omites |
|---|---|---|---|
| **C** | **Contexto** | Qué datos existen ya, cómo se llaman las variables y las columnas | El asistente inventa nombres de columnas y el código falla |
| **L** | **Librerías** | Con qué debe resolverlo, y qué no debe instalar | Te devuelve algo con una librería que no está en Colab |
| **A** | **Acción** | Qué tiene que hacer, en términos del negocio | Te da lo técnicamente correcto para otra pregunta |
| **R** | **Resultado** | En qué formato lo quieres de vuelta | Te entrega tres páginas de prosa y el código enterrado |
| **O** | **Objeción** | Qué supuestos debe declarar antes de responder | No te enteras de las decisiones que tomó por ti |

La letra que más alumnos se saltan es la **O**. Es la que convierte al asistente en alguien que
discute contigo, en vez de alguien que te da la razón.

### El mismo encargo, mal y bien

Así pide el 90 % de la gente:

```text
dame el código para ver las ventas por ciudad
```

Devuelve algo. Casi seguro con nombres de columna inventados, casi seguro sin decirte que tuvo que
decidir qué hacer con las filas cuyo cliente no existe en el padrón. Ahora el mismo encargo en CLARO:

```text
CONTEXTO: en Colab tengo un DataFrame `ventas` con la columna cliente_id y otro
`clientes` con cliente_id y ciudad. No todos los cliente_id de ventas existen en
clientes.
LIBRERÍAS: pandas, ya importado como pd. No instales nada.
ACCIÓN: sumar la facturación por ciudad usando un merge por la izquierda desde
ventas hacia clientes.
RESULTADO: una sola celda de código comentada, sin prosa.
OBJECIÓN: antes del código, dime en una línea qué pasa con las filas de ventas
cuyo cliente_id no aparece en clientes.
```

El segundo encargo es más largo. También es el único de los dos cuyo resultado puedes defender en un
comité. **La longitud no es la virtud: la virtud es que no queda nada decidido por omisión.**

## 2. La bitácora, desde el primer encargo

La bitácora no es papeleo de cierre: se llena mientras trabajas. Es el 10 % de la nota de cada deber
y es lo que te permite responder «¿por qué hiciste esto?» tres semanas después.

Ejecuta la celda para tener disponibles `registrar()` —que añade una fila— y `comprobar()` —que
contrasta un número tuyo contra el valor verificado del curso.

In [ ]:
bitacora = pd.DataFrame(columns=[
    "fecha", "objetivo", "prompt_final", "que_devolvio",
    "como_lo_verifique", "veredicto",
])

def registrar(objetivo, prompt_final, que_devolvio, como_lo_verifique, veredicto):
    """Añade una fila a la bitácora. veredicto: 'acepto', 'corrijo' o 'descarto'."""
    global bitacora
    fila = pd.DataFrame([{
        "fecha": pd.Timestamp.today().date(),
        "objetivo": objetivo,
        "prompt_final": prompt_final,
        "que_devolvio": que_devolvio,
        "como_lo_verifique": como_lo_verifique,
        "veredicto": veredicto,
    }])
    bitacora = pd.concat([bitacora, fila], ignore_index=True)
    return bitacora

def comprobar(etiqueta, obtenido, esperado, tolerancia=0.01):
    """Contrasta un número tuyo contra el valor verificado del curso."""
    try:
        obtenido = float(obtenido)
    except (TypeError, ValueError):
        print(f"❔ {etiqueta}: no pude leer un número. ¿Ejecutaste la celda anterior?")
        return False
    bien = abs(obtenido - esperado) <= tolerancia
    print(f"{'✅' if bien else '❌'} {etiqueta}")
    print(f"   tuyo {obtenido:>16,.2f}   esperado {esperado:>16,.2f}")
    return bien

print("Listo: usa registrar(...) y comprobar(...) durante todo el cuaderno.")

## 3. Cargar: los primeros cinco minutos con una tabla nueva

Comercial Andina es un distribuidor ecuatoriano con tiendas en Quito, Guayaquil, Cuenca, Manta y
Loja, más un canal en línea. Sus datos son los mismos las dieciséis semanas del curso.

### 📋 Encargo 1 — Cargar y medir

```text
CONTEXTO: trabajo en Google Colab. En el cuaderno ya existe una variable `DATOS`
de tipo pathlib.Path que apunta a la carpeta con los CSV del curso. Dentro hay
ventas.csv y clientes.csv. ventas.csv tiene las columnas factura_id, fecha,
cliente_id, sucursal_id, producto_id, cantidad, precio_unitario, descuento,
es_devolucion.
LIBRERÍAS: pandas, ya importado como pd. No instales nada ni uses otra librería.
ACCIÓN: cargar los dos archivos en los DataFrames `ventas` y `clientes`, e
imprimir cuántas filas y columnas tiene cada uno.
RESULTADO: una sola celda de código con comentarios cortos. Sin prosa antes ni
después del bloque de código.
OBJECIÓN: antes del código, dime en una línea qué estás suponiendo sobre el
separador y la codificación de los archivos.
```

In [ ]:
# 📋 Pega aquí el código del Encargo 1

In [ ]:
# --- Comprobación del Encargo 1 (esta celda ya viene escrita) ---
print(f"ventas   : {ventas.shape[0]:,} filas × {ventas.shape[1]} columnas")
print(f"clientes : {clientes.shape[0]:,} filas × {clientes.shape[1]} columnas")
print()
comprobar("filas de ventas",   ventas.shape[0],   80_515, tolerancia=0)
comprobar("filas de clientes", clientes.shape[0],  1_800, tolerancia=0)

📌 Ochenta mil filas y mil ochocientos clientes. Ojo con lo que representa **una fila**: no es una
venta ni un cliente, es **una línea de una factura**. Un cliente que compró tres productos genera
tres filas. Confundir la unidad de análisis es el error que abre la semana 2, y ya te puede costar
este laboratorio.

Si te salió otro número de columnas, tu asistente probablemente adivinó un separador distinto.

## 4. La columna que no viene en el archivo

`ventas.csv` no trae una columna de dinero: trae cantidad, precio y descuento. El monto hay que
construirlo, y ahí ya hay una decisión que tomar.

Este encargo es el más importante del cuaderno y por eso es el más específico. Fíjate en que **le
prohíbe expresamente arreglar los nulos**: hay 322 filas con celdas vacías, y lo que se hace con
ellas es materia de la semana 4. Si dejas esa decisión abierta, cada asistente elegirá una cosa
distinta y ninguno de tus números coincidirá con los del resto de la clase.

### 📋 Encargo 2 — Fecha y monto

```text
CONTEXTO: en Colab tengo un DataFrame `ventas` con las columnas fecha, cantidad,
precio_unitario, descuento, factura_id y cliente_id. La columna fecha viene con
formatos mezclados: unas filas como d/m/aaaa y otras como aaaa-mm-dd. Hay 322
filas con celdas vacías en cantidad, precio_unitario o descuento.
LIBRERÍAS: pandas, ya importado como pd. No instales nada.
ACCIÓN:
  1. Convertir ventas["fecha"] a datetime usando EXACTAMENTE
     pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True).
  2. Crear la columna ventas["monto"] como
     cantidad × precio_unitario × (1 − descuento).
  3. NO rellenar, NO eliminar y NO imputar los valores nulos: déjalos como NaN.
  4. Imprimir la fecha mínima y la máxima, la facturación total, el número de
     facturas únicas y cuántos clientes distintos compraron.
RESULTADO: una sola celda de código comentada, sin prosa.
OBJECIÓN: si crees que hay una forma mejor de tratar los nulos o de parsear la
fecha, escríbela en una línea pero NO la apliques al código.
```

In [ ]:
# 📋 Pega aquí el código del Encargo 2

In [ ]:
# --- Comprobación del Encargo 2 (esta celda ya viene escrita) ---
comprobar("facturación total", ventas["monto"].sum(), 2_841_509.48)
comprobar("facturas únicas",   ventas["factura_id"].nunique(), 17_675, tolerancia=0)
comprobar("clientes que compraron", ventas["cliente_id"].nunique(), 1_751, tolerancia=0)
comprobar("líneas sin monto",  ventas["monto"].isna().sum(), 322, tolerancia=0)
print(f"\nperiodo: {ventas['fecha'].min():%d-%m-%Y} a {ventas['fecha'].max():%d-%m-%Y}")

Si la facturación total no te dio **2 841 509,48**, casi siempre es por una de tres razones, en este
orden de frecuencia:

1. El asistente «arregló» los nulos rellenándolos con cero o con la media, pese a que se lo
   prohibiste. Vuelve a leer lo que te devolvió: es el fallo más común y el más silencioso.
2. Interpretó el descuento como porcentaje entero (25) en vez de proporción (0,25).
3. Parseó las fechas con el formato por defecto y perdió filas por el camino.

**Registra ahora este encargo en la bitácora**, con veredicto `acepto` si pasó a la primera y
`corrijo` si tuviste que reescribirlo. Esta es la primera de tus tres filas obligatorias.

## 5. El pulso del negocio: treinta meses en una línea

### 📋 Encargo 3 — La serie mensual

```text
CONTEXTO: en Colab tengo el DataFrame `ventas` con la columna fecha ya en
datetime y la columna monto ya calculada.
LIBRERÍAS: pandas, ya importado como pd. No instales nada.
ACCIÓN: agrupar ventas por mes calendario a partir de la columna fecha, sumar
monto y guardar el resultado en una Series llamada `mensual` cuyo índice sea de
tipo timestamp (no Period). Imprimir los tres últimos meses.
RESULTADO: una sola celda de código comentada, sin prosa.
OBJECIÓN: antes del código, dime en una línea si los meses sin ninguna venta
aparecerían o no en el resultado.
```

In [ ]:
# 📋 Pega aquí el código del Encargo 3

In [ ]:
# --- Comprobación del Encargo 3 (esta celda ya viene escrita) ---
print(mensual.tail(3).to_string())
print()
comprobar("julio de 2026", mensual.iloc[-1], -776.80)

⚠️ Julio de 2026 cierra en **−776,80**. No es una caída del negocio ni un error del asistente: el
archivo se corta el 18 de julio y ese mes solo alcanzó a registrar devoluciones netas.

Un mes incompleto metido en una serie temporal es la forma más barata de inventar una tendencia a la
baja que no existe. Se recorta, y **se dice que se recortó**.

Fíjate en algo: tu asistente no podía saberlo. Tenía los datos correctos y la instrucción correcta, y
aun así el resultado, presentado sin esta nota, sería engañoso. Eso no lo arregla un prompt mejor: lo
arregla alguien que conoce el negocio mirando el número.

### 📋 Encargo 4 — El gráfico honesto

```text
CONTEXTO: en Colab tengo una Series `mensual` con la facturación por mes,
índice de tipo timestamp, desde enero de 2024. El último punto es julio de 2026
y está incompleto, así que hay que excluirlo.
LIBRERÍAS: pandas y matplotlib.pyplot como plt, ya importados. seaborn ya tiene
el tema aplicado. No instales nada.
ACCIÓN: crear `serie` como mensual sin los meses de julio de 2026 en adelante,
graficarla como línea con marcadores, y después imprimir el mes más alto con su
valor, el promedio mensual y cuántas veces diciembre de 2025 fue mayor que
febrero de 2025.
RESULTADO: una sola celda de código comentada. El eje y etiquetado en español y
sin título inventado: el título lo pongo yo.
OBJECIÓN: antes del código, dime en una línea por qué recortar el último mes
puede ser también una forma de manipular un gráfico.
```

In [ ]:
# 📋 Pega aquí el código del Encargo 4

In [ ]:
# --- Comprobación del Encargo 4 (esta celda ya viene escrita) ---
comprobar("meses en la serie", len(serie), 30, tolerancia=0)
comprobar("mes más alto (dic-2025)", serie.max(), 136_701.23)
comprobar("promedio mensual", serie.mean(), 94_742.88)
print(f"\ndiciembre-2025 sobre febrero-2025: "
      f"{serie['2025-12-01'] / serie['2025-02-01']:.2f} veces   (esperado 1,86)")

Diciembre es el pico del año: **136,7 mil** frente a los **94,7 mil** del mes promedio. Hay
estacionalidad y es fuerte. Guárdala: es la razón por la que en la semana 12 no se puede pronosticar
con una recta sin más.

La pregunta de la **O** de este encargo tiene respuesta y es incómoda: recortar el último mes es
legítimo aquí porque está incompleto, pero exactamente el mismo gesto sirve para esconder un mes malo.
La diferencia entre las dos cosas no está en el código, está en si lo declaraste.

## 6. El informe de apertura: datos correctos, conclusión inválida

Lo que sigue es un informe real, reescrito con los datos de Comercial Andina. Lo firmó un analista
competente, con datos correctos y software correcto. La conclusión es falsa igual.

La pregunta que le hicieron fue: *¿cuál es nuestra mejor ciudad?*

### 📋 Encargo 5 — Reproducir el informe

```text
CONTEXTO: en Colab tengo `ventas` (con columnas cliente_id y monto) y
`clientes` (con cliente_id, ciudad y tipo_cliente). La columna ciudad viene
sucia: hay espacios sobrantes, mayúsculas inconsistentes y "Guayaquíl" con
tilde. Además, no todos los cliente_id de ventas existen en clientes.
LIBRERÍAS: pandas y matplotlib.pyplot como plt, ya importados. No instales nada.
ACCIÓN:
  1. Limpiar clientes["ciudad"] con .str.strip().str.title() y reemplazar
     "Guayaquíl" por "Guayaquil". Sin tocar nada más de esa columna.
  2. Unir ventas con clientes[["cliente_id","ciudad","tipo_cliente"]] mediante
     un merge por la izquierda (how="left") sobre cliente_id, en un DataFrame `v`.
  3. Sumar el monto por ciudad, ordenado de mayor a menor, en `por_ciudad`.
  4. Graficarlo como barras e imprimir la cuota de Quito y de Loja sobre la
     facturación total, y cuántas veces Quito es mayor que Guayaquil y que Loja.
RESULTADO: una sola celda de código comentada, sin prosa.
OBJECIÓN: antes del código, dime en una línea cuánta facturación queda fuera de
`por_ciudad` por pertenecer a clientes que no están en el padrón.
```

In [ ]:
# 📋 Pega aquí el código del Encargo 5

In [ ]:
# --- Comprobación del Encargo 5 (esta celda ya viene escrita) ---
total = ventas["monto"].sum()
comprobar("Quito, facturación", por_ciudad["Quito"], 958_688.61)
comprobar("Loja, facturación",  por_ciudad["Loja"],  200_145.44)
print(f"\ncuota de Quito : {por_ciudad['Quito'] / total * 100:.1f} %   (esperado 33,7 %)")
print(f"cuota de Loja  : {por_ciudad['Loja']  / total * 100:.1f} %   (esperado 7,0 %)")
print(f"Quito / Loja   : {por_ciudad['Quito'] / por_ciudad['Loja']:.2f} veces   (esperado 4,79)")
print(f"\nfacturación sin ciudad (clientes fuera del padrón): "
      f"{total - por_ciudad.sum():,.2f}  =  {(total - por_ciudad.sum()) / total * 100:.1f} %")

Ese 7,5 % que se queda fuera es la respuesta a la **O** del encargo: son ventas de clientes que no
existen en el padrón. Si tu asistente no te lo advirtió antes de escribir el código, tu encargo
funcionó pero tu asistente no te está cuidando. Anótalo.

Y aquí está la conclusión que llegó al comité, textual:

> «Quito es nuestra mejor plaza: concentra el 33,7 % de la facturación total, 1,38 veces lo de
> Guayaquil y 4,79 veces lo de Loja. Recomendamos reforzar la fuerza comercial de Quito y revisar la
> continuidad de Loja, que apenas aporta el 7,0 %.»

Los números son correctos. Acabas de verificarlos tú mismo. La conclusión, en cambio, no se sostiene:
**la pregunta era cuál es la mejor ciudad y lo que se midió fue cuál es la más grande.**

## 7. La corrección: dividir por lo que hace grande a la ciudad

Quito tiene más clientes que Loja. Que facture más no dice nada sobre su desempeño, igual que un curso
de sesenta alumnos no es mejor que uno de veinte por tener más aprobados. La comparación honesta divide
por el tamaño de la base.

### 📋 Encargo 6 — La métrica que sí compara

```text
CONTEXTO: en Colab tengo la Series `por_ciudad` con la facturación por ciudad y
el DataFrame `clientes` con la columna ciudad ya limpia.
LIBRERÍAS: pandas y matplotlib.pyplot como plt, ya importados. No instales nada.
ACCIÓN:
  1. Construir un DataFrame `tabla` con dos columnas: "ventas" (desde
     por_ciudad) y "clientes" (el número de clientes de cada ciudad según el
     padrón completo, no solo los que compraron).
  2. Añadir "ventas_por_cliente" = ventas / clientes y ordenar por esa columna
     de mayor a menor.
  3. Graficarla como barras e imprimir la cifra de Cuenca, la de Quito y qué
     porcentaje rinde Cuenca por encima de Quito.
RESULTADO: una sola celda de código comentada, sin prosa.
OBJECIÓN: antes del código, dime en una línea qué diferencia habría si el
denominador fueran solo los clientes que compraron al menos una vez.
```

In [ ]:
# 📋 Pega aquí el código del Encargo 6

In [ ]:
# --- Comprobación del Encargo 6 (esta celda ya viene escrita) ---
comprobar("Cuenca, ventas por cliente", tabla.loc["Cuenca", "ventas_por_cliente"], 1_877.62)
comprobar("Quito, ventas por cliente",  tabla.loc["Quito",  "ventas_por_cliente"], 1_422.39)

r = tabla.loc["Cuenca", "ventas_por_cliente"] / tabla.loc["Quito", "ventas_por_cliente"]
print(f"\nCuenca rinde un {(r - 1) * 100:.1f} % más por cliente   (esperado 32,0 %)")

brecha = tabla.loc["Cuenca", "ventas_por_cliente"] - tabla.loc["Quito", "ventas_por_cliente"]
oportunidad = brecha * tabla.loc["Quito", "clientes"]
print(f"Si cada cliente de Quito rindiera como uno de Cuenca, Quito facturaría "
      f"{tabla.loc['Quito', 'ventas'] + oportunidad:,.2f} en lugar de {tabla.loc['Quito', 'ventas']:,.2f}")
print(f"Oportunidad no capturada en Quito : {oportunidad:,.2f}   (esperado 306 829,69)")
print(f"Equivale al {oportunidad / ventas['monto'].sum() * 100:.1f} % de la facturación total")

📌 El informe original recomendaba **meter más comercial en Quito**. El análisis corregido dice lo
contrario: en Quito ya hay clientes de sobra y lo que falla es cuánto compra cada uno. La acción no es
captar; es entender qué hace Cuenca con su base y replicarlo. Mismo dato, decisión opuesta, **306 829,69
sobre la mesa**.

Y ahora lo que importa para este cuaderno: **el asistente habría producido los dos análisis con la
misma diligencia.** Ninguno de los seis encargos era técnicamente más difícil que el otro. Lo único que
cambió fue cuál pregunta se hizo.

## 8. Protocolo de uso de IA generativa

En este curso el asistente se usa desde hoy y para todo. Lo que se califica no es el código que
devuelve sino **la calidad del encargo y el rigor de la verificación**. El protocolo tiene tres reglas:

1. **Puedes pedir** código, explicaciones, alternativas y crítica de tu propio razonamiento.
2. **Tienes que verificar** todo número que salga del asistente contra una ejecución tuya. En este
   cuaderno te lo dieron hecho con `comprobar()`; a partir de la semana 2 el contraste lo diseñas tú.
3. **Tienes que registrar** el prompt final y una línea sobre qué comprobaste. Eso es la bitácora.

Esta es una fila bien escrita, para que veas el nivel de detalle que se espera:

In [ ]:
registrar(
    objetivo="Reproducir el informe de ciudades sin que el merge pierda filas en silencio",
    prompt_final="CONTEXTO: ventas con cliente_id y monto, clientes con cliente_id y ciudad sucia; "
                 "no todos los cliente_id de ventas existen en clientes. LIBRERÍAS: pandas. "
                 "ACCIÓN: limpiar ciudad, merge how='left' y sumar monto por ciudad. "
                 "RESULTADO: una celda comentada. OBJECIÓN: cuánta facturación queda fuera.",
    que_devolvio="El merge correcto, pero solo mencionó las filas huérfanas cuando se lo pedí en la O",
    como_lo_verifique="Comparé por_ciudad.sum() contra ventas['monto'].sum(): faltaba el 7,5 %",
    veredicto="corrijo",
)

### 🌶️ Ejercicio 1 — Guiado

El informe comparó ciudades. Repite exactamente la misma trampa y la misma corrección con
**`tipo_cliente`**: primero la facturación total de mayoristas frente a minoristas, después la
facturación por cliente de cada grupo.

Aquí el encargo viene con huecos. **Complétalos tú** antes de pegarlo en el asistente:

```text
CONTEXTO: en Colab tengo el DataFrame `v` (ventas unido con clientes) que
incluye las columnas monto y tipo_cliente, y el DataFrame `clientes` con
cliente_id y tipo_cliente. Los valores de tipo_cliente son ______ y ______.
LIBRERÍAS: ______________________________________
ACCIÓN: ________________________________________
        ________________________________________
RESULTADO: ______________________________________
OBJECIÓN: _______________________________________
```

Después escribe **en una frase** qué cambia y qué no cambia respecto de lo que pasó con las ciudades.
Ojo: la respuesta aquí no es la misma que arriba, y esa es exactamente la gracia del ejercicio.

In [ ]:
# 📋 Pega aquí el código del Ejercicio 1

### 🔥 Desafío

`ventas_por_cliente` divide por **todos** los clientes del padrón, incluidos los que nunca compraron.
Construye una tercera columna, `ventas_por_cliente_activo`, que divida solo entre los clientes que sí
facturaron, y responde: ¿cambia el ranking de ciudades? ¿Cuál de las dos métricas defenderías ante el
gerente comercial y por qué?

Esta vez **el encargo lo escribes entero desde cero**, con las cinco letras. Pégalo en la celda de
abajo como comentario, encima del código que te devuelva el asistente, para que quede en la entrega.

In [ ]:
# 📋 Tu encargo CLARO (como comentario) y debajo el código del asistente

### 🎯 Reto en clase (15 min)

En parejas. Toma la conclusión del informe («reforzar Quito, revisar la continuidad de Loja») y
escribe **la versión corregida en tres frases**, con una cifra en cada una, lista para un comité que
no va a ver tu código.

Después pídele al asistente que la ataque: dale el rol del gerente de Quito defendiendo su
presupuesto y pídele las tres objeciones más fuertes que puede hacerle a tu conclusión. Responde al
menos una con un número que puedas calcular. Registra el intercambio completo en la bitácora.

In [ ]:
# 📋 Tu conclusión en tres frases, las objeciones del gerente y tu respuesta con datos

## La trampa de hoy

⚠️ **Creer que un asistente mejor te habría salvado del informe.** El informe de apertura se hizo con
datos correctos, software correcto y aritmética correcta. Hoy lo reprodujiste encargándoselo a un
modelo de última generación y salió idéntico: correcto y equivocado.

Pruébalo tú mismo. Copia esto en tu asistente, sin más contexto que el que lleva:

```text
Tengo las ventas de una empresa por ciudad: Quito 958 688, Guayaquil 695 070,
Cuenca 484 427, Manta 290 337, Loja 200 145. ¿Cuál es mi mejor ciudad?
```

Va a contestar Quito. Con seguridad, con porcentajes y probablemente con una recomendación comercial.
No porque sea un mal modelo, sino porque **la palabra «mejor» no estaba definida en el encargo y la
definió por ti**: eligió «la más grande», que es la definición que la pregunta sugiere.

Ese es el fallo del informe original, y ninguna versión más nueva de nada lo arregla. Los dos números
lado a lado:

In [ ]:
comparacion = pd.DataFrame({
    "métrica del informe (total facturado)": por_ciudad,
    "métrica correcta (por cliente)": tabla["ventas_por_cliente"],
})
comparacion["puesto informe"] = comparacion.iloc[:, 0].rank(ascending=False).astype(int)
comparacion["puesto correcto"] = comparacion.iloc[:, 1].rank(ascending=False).astype(int)
comparacion = comparacion.sort_values("puesto informe")

print("Mejor ciudad según el informe         :", comparacion.index[0])
print("Mejor ciudad según la métrica correcta:", comparacion.sort_values("puesto correcto").index[0])
comparacion

Las dos columnas salen del mismo archivo, de la misma máquina y del mismo asistente. Lo único que
cambia es el denominador, y el denominador lo pusiste tú en la **A** del encargo. Si el gerente
hubiera preguntado *«¿comparado con qué?»* —la pregunta que abre la semana 9— el informe no habría
salido de la sala.

**La conclusión del laboratorio, en una línea:** el asistente es excelente ejecutando y no tiene
opinión sobre si tu pregunta vale algo. Esa parte sigue siendo tuya, y es la que se califica.

## Entregable

Sube `lab_01_ia_apellido.ipynb` con:

- El cuaderno ejecutado de arriba a abajo, con los seis encargos pegados y todas las comprobaciones
  en ✅. Si alguna quedó en ❌, deja una celda de texto explicando qué pasó: **un ❌ explicado vale
  más que un ✅ que no sabes por qué salió**.
- Los tres ejercicios resueltos, cada uno con su encargo CLARO visible y su frase de interpretación.
- La bitácora con **al menos tres filas propias**, una de ellas con veredicto `corrijo` o `descarto`.
- Al final, en una celda de texto: de los seis encargos, **cuál habría dado un resultado distinto si
  le hubieras quitado la letra O**. Uno solo, y explica qué se habría colado.

Ejecuta la última celda para ver tu bitácora antes de entregar.

In [ ]:
bitacora

## Para tu equipo

- Firmen el contrato de grupo y abran el repositorio compartido esta misma semana. La bitácora de
  prompts vive ahí desde hoy, no desde la semana en que se entrega el proyecto.
- Elijan una empresa cuyos datos puedan conseguir de verdad. Un caso brillante sin datos se convierte
  en un ensayo, y este curso no califica ensayos.
- Busquen en su empresa el equivalente del informe de hoy: un indicador que se reporta en total cuando
  debería reportarse por unidad (por cliente, por tienda, por vendedor, por metro cuadrado). Lo van a
  encontrar. Anótenlo: es el punto de partida del proyecto integrador.
- Acuerden ya el formato de la bitácora del grupo. Seis semanas de prompts sin registrar no se
  reconstruyen de memoria la noche antes de la entrega.